## Proof-of-concept:

#### A) use of a 'flexible' classifai-provided hook for a common task (capitalisation control)

#### B) flexible use of a classifai-provided RAG-type hook (classification, re-ranking, keyword identification)

In [ ]:
from classifai.indexers import VectorStore
from classifai.indexers.dataclasses import VectorStoreSearchInput
from classifai.indexers.default_hooks.pre_processing import CapitalisationStandardisingHook
from classifai.indexers.default_hooks.rag import RAGHook
from classifai.vectorisers import GcpVectoriser

# Initialise the vectoriser
demo_vectoriser = GcpVectoriser(project_id="YOUR-PROJECT-ID", vertexai=True)


def capitalise_candidate_texts(input_data):
    # convert the result texts to CAPITAL letters
    input_data["doc_text"] = input_data["doc_text"].str.upper()
    print(type(input_data))
    return input_data


default_hook_capitalise = CapitalisationStandardisingHook(method="upper")

# CLASSIFICATION
CONTEXT_PROMPT_CLASSIFICATION = """You are an agent helping classify job descriptions.
Your task is to identify the most relevant retrieved document for a given query.
You will receive a Pandas DataFrame, with a schema described in an Input Format
section, which is converted to JSON.
You must identify the row with the most relevant retrieved document.
When you have identified the most relevant row, respond strictly in the format described
in the Output Format section.
"""
RESPONSE_TEMPLATE_CLASSIFICATION = """
Respond ONLY with a single JSON string, which is a list of boolean values the same length as
the DataFrame you received - in the JSON list, put a "1" at the element corresponding to the
most relevant row, and "0"s for all other elements.
Make sure to avoid backticks, using only single or double quotes throughout.
Ensure the square brackets defining the list start and end your response, do not deviate
from this. Take the format of the example below as a strict guide.
Example (in the case where there are 5 rows, and the second is the most relevant):
["0","1","0","0","0"]
"""

# RE-RANKING
CONTEXT_PROMPT_RERANKING = """You are an agent helping classify job descriptions.
Your task is to rank the retrieved documents for a given query in order of relevance.
You will receive a Pandas DataFrame, with a schema described in an Input Format
section, which is converted to JSON.
You must identify the row with the most relevant retrieved document.
When you have identified the most relevant row, respond strictly in the format described
in the Output Format section.
"""
RESPONSE_TEMPLATE_RERANKING = """
Respond ONLY with a list of boolean values the same length as the DataFrame you received -
in the list, put a "1" at the element corresponding to the most relevant row, "2" at the 
next most relevant, and so on for all elements.
Make sure to avoid backticks, using only single or double quotes throughout.
Ensure the square brackets defining the list start and end your response, do not deviate
from this. Take the format of the example below as a strict guide.
Example (in the case where there are 5 rows, and the second is the most relevant):
["4","1","3","2","5"]
"""

# KEYWORD IDENTIFICATION
CONTEXT_PROMPT_KEYWORD_INDENTIFICATION = """You are an agent helping classify job descriptions.
Your task is to identify the keyword within each of the retrieved documents which is
most relevant to the given query.
You will receive a Pandas DataFrame, with a schema described in an Input Format
section, which is converted to JSON.
You must identify the most relevant keyword within each document text.
When you have identified the most relevant keyword, respond strictly in the format described
in the Output Format section.
"""
RESPONSE_TEMPLATE_KEYWORD_INDENTIFICATION = """
Respond ONLY with a single JSON string, which is a list of boolean values the same length as
the DataFrame you received - in the JSON list, there should be "keyword_1" as the first element
(where keyword_1 is in the first row's document text and is the most relevant to the query),
"keyword_2" as the next element, and so on for all elements.
Make sure to avoid backticks, using only single or double quotes throughout.
Ensure the square brackets defining the list start and end your response, do not deviate
from this. Take the format of the example below as a strict guide.
Example (in the case where there are 5 rows, and the query describes royalty):
["king","knight","throne","crown","castle"]
"""

rag_hook = RAGHook(
    project_id="YOUR-PROJECT-ID",
    vertexai=True,
    context_prompt=CONTEXT_PROMPT_KEYWORD_INDENTIFICATION,
    response_template=RESPONSE_TEMPLATE_KEYWORD_INDENTIFICATION,
)

# Initialise the vector store, pointing the demo to the demo test data
demo_vectorstore = VectorStore(
    file_name="./data/fake_soc_dataset.csv",
    data_type="csv",
    vectoriser=demo_vectoriser,
    output_dir="./demo_vdb",
    overwrite=True,
    hooks={"search_preprocess": default_hook_capitalise, "search_postprocess": rag_hook},
)


query_df = VectorStoreSearchInput({"id": [1, 2], "query": ["apple merchant", "pub landlord"]})

demo_vectorstore.search(query_df, n_results=5)

INFO - Processing file: ./data/fake_soc_dataset.csv...

100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [00:06<00:00,  3.45it/s]
INFO - Gathering metadata and saving vector store / metadata...
INFO - Vector Store created - files saved to ./demo_vdb
Processing query batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  8.87it/s]
INFO - AFC is enabled with max remote calls: 10.
INFO - AFC is enabled with max remote calls: 10.


,query_id,query_text,doc_id,doc_text,rank,score,RAG_response
0,1,APPLE MERCHANT,101,Fruit farmer: Grows and harvests fruits such a...,0,0.652508,apples
1,1,APPLE MERCHANT,101,Fruit farmer: Grows and harvests fruits such a...,1,0.652508,apples
2,1,APPLE MERCHANT,101,Orchard worker: Maintains and harvests fruit t...,2,0.635190,fruit trees
3,1,APPLE MERCHANT,107,Mobile app developer: Creates applications for...,3,0.620049,app
4,1,APPLE MERCHANT,128,"Real estate agent: Assists clients in buying, ...",4,0.618706,selling
5,2,PUB LANDLORD,138,Bartender: Mixes and serves drinks to customers.,0,0.623973,Bartender
6,2,PUB LANDLORD,128,Property manager: Oversees the maintenance and...,1,0.610828,Property manager
7,2,PUB LANDLORD,129,Venue manager: Manages the operations of event...,2,0.608057,Venue manager
8,2,PUB LANDLORD,112,Bartender: Prepares and serves drinks in bars ...,3,0.603574,Bartender
9,2,PUB LANDLORD,138,Mixologist: Creates unique and innovative cock...,4,0.591556,Mixologist
